# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata object
metadata = dataset.metadata

print("Dataset title:", metadata.name)
print("Description:", metadata.description)
print("Published:", metadata.datePublished)
print("Identifier:", metadata.identifier)
print("Keywords:", metadata.keywords)


## 2. Data Overview
Review the available record sets, fields, and their IDs. All references are made using their `@id` values.

In [ ]:
# Retrieve the record sets in the dataset
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    # recordSet can be list of objects
    for rs in metadata.recordSet:
        print("RecordSet @id:", rs['@id'])
        record_sets.append(rs['@id'])
else:
    print("No record sets found in metadata.")

if record_sets:
    # Examine available fields/columns in each record set
    for rs_id in record_sets:
        print(f"\n---- Records for RecordSet {rs_id} ----")
        for i, rec in enumerate(dataset.records(record_set=rs_id)):
            print(f"Record {i}: {rec}")
            if i == 2:  # print up to 3 samples per set
                break


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare to extract all record sets as DataFrames
dataframes = {}

if record_sets:
    for rs_id in record_sets:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nDataFrame columns for RecordSet {rs_id}:")
        print(df.columns.tolist())
        print(df.head())
else:
    print("No record sets to extract.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, and grouping. Use column `@id` values for referencing fields.

In [ ]:
# EDA examples (assuming record sets and columns based on typical medical/clinical tables)
# Replace examples below with actual @id references as discovered
example_record_set_id = record_sets[0] if record_sets else None
example_df = dataframes.get(example_record_set_id, pd.DataFrame())

# Suppose a numeric field with @id 'cr:age_at_diagnosis', categorical 'cr:hirsutism'
numeric_field_id = 'cr:age_at_diagnosis'
group_field_id = 'cr:hirsutism'

if not example_df.empty and numeric_field_id in example_df:
    # Filter records with age_at_diagnosis > 25
    threshold = 25
    filtered_df = example_df[example_df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[numeric_field_id + '_normalized'] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Group by categorical field if present
    if group_field_id in filtered_df:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Cannot perform EDA: {numeric_field_id} not present or dataframe is empty.")

## 5. Visualization
Visualize distributions or relationships between numeric and categorical fields. Use field and record set `@id` references.

In [ ]:
# Visualization example with matplotlib
import matplotlib.pyplot as plt

if not example_df.empty and numeric_field_id in example_df:
    plt.figure(figsize=(7,4))
    plt.hist(example_df[numeric_field_id].dropna(), bins=5, color='skyblue', edgecolor='black')
    plt.xlabel('Age at Diagnosis (@id: cr:age_at_diagnosis)')
    plt.ylabel('Count')
    plt.title('Distribution of Age at Diagnosis')
    plt.show()

    # Boxplot for numeric field grouped by categorical field (if present)
    if group_field_id in example_df:
        plt.figure(figsize=(7,4))
        example_df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.xlabel('Hirsutism (@id: cr:hirsutism)')
        plt.ylabel('Age at Diagnosis (@id: cr:age_at_diagnosis)')
        plt.title('Age at Diagnosis grouped by Hirsutism')
        plt.suptitle('')
        plt.show()
else:
    print("No visualization: numeric/categorical fields missing or DataFrame empty.")

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR^2 clinical dataset using the `mlcroissant` library. Data were reviewed using unique `@id` references for record sets and fields, enabling reproducible access and manipulation. Limited cohort size and clinical scope were observed; nonetheless, the structured schema supports analytic workflows and visualization suitable for academic research and case-based clinical study.